# Agent Failures — Quicklook

Load the public JSONL, plot failure taxonomy, and look up any trace by ID.

**Dataset:** [github.com/lakmus-ai/agent-failures](https://github.com/lakmus-ai/agent-failures) · **HF (v1):** [lakmus/agent-failures-v1](https://huggingface.co/datasets/lakmus/agent-failures-v1)

**Docs:** [METHODOLOGY.md](https://github.com/lakmus-ai/agent-failures/blob/main/docs/METHODOLOGY.md) (paired vs raw stats) · [DATASET.md](https://github.com/lakmus-ai/agent-failures/blob/main/docs/DATASET.md)

Run all cells after cloning the repo (paths assume `notebooks/` as cwd).

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

REPO = Path("..").resolve()  # run from notebooks/
DATA = REPO / "data"

def load_jsonl(name: str) -> list[dict]:
    path = DATA / name
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

failures = load_jsonl("failures-v1.jsonl")
passes = load_jsonl("passes-v1.jsonl")
paired_stats = json.loads((DATA / "stats-paired-v1.json").read_text(encoding="utf-8"))

print(f"Failures: {len(failures)} | Passes: {len(passes)} | Paired tasks: {paired_stats['paired_task_count']}")

In [ ]:
# Paired leaderboard (fair comparison — same 49 tasks, all 7 models)
rows = sorted(
    paired_stats["model_comparison"].items(),
    key=lambda x: -x[1]["success_rate"],
)
for model, s in rows:
    pct = s["success_rate"] * 100
    print(f"{model:28} {s['passed']:2}/{s['paired_tasks']}  ({pct:.0f}%)")

In [ ]:
# Failure taxonomy chart
counts = Counter(r["failure_type"] for r in failures)
labels, values = zip(*counts.most_common()) if counts else ([], [])

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels[::-1], values[::-1], color="#6366f1")
ax.set_xlabel("Count")
ax.set_title("Failure types (v1 paired benchmark)")
plt.tight_layout()
plt.show()

In [ ]:
# Trace lookup — comfort zone reversion (DeepSeek)
TRACE_ID = "A0D678B"

record = next((r for r in failures if r["trace_id"] == TRACE_ID), None)
if record is None:
    print(f"Trace {TRACE_ID} not in failures-v1.jsonl")
else:
    print("trace_id:", record["trace_id"])
    print("model:", record["model_display_name"])
    print("failure:", record["failure_type"], "/", record["failure_subtype"])
    print("task:", record["task"][:200], "...")
    print("\n--- final_answer (first 800 chars) ---\n")
    print((record.get("final_answer") or "")[:800])